# w2v-bert-tpu: inference demo

Runs [`facebook/w2v-bert-2.0`](https://huggingface.co/facebook/w2v-bert-2.0) on a TPU v5e with
[w2v-bert-tpu](https://github.com/PositiveLoss/w2v-bert-tpu), using the example clips from the
[Yehor/w2v-bert-uk-v3](https://huggingface.co/spaces/Yehor/w2v-bert-uk-v3) Space.

**Runtime:** Colab TPU v5e with TorchTPU (`torch_tpu`) preinstalled. Run the cells from a fresh
runtime, top to bottom.

The model outputs hidden states (`[batch, frames, 1024]`), not text. It is the pretrained
encoder, without a speech-recognition head.

## 1. Install

In [ ]:
import importlib.metadata as md
import subprocess
import sys


def version(pkg):
    try:
        return md.version(pkg)
    except md.PackageNotFoundError:
        return None


# Clone the code.
subprocess.run(["git", "clone", "-q", "https://github.com/PositiveLoss/w2v-bert-tpu"], check=False)

# Python packages needed for inference.
packages = ["transformers", "soundfile", "huggingface_hub"]
# jax compiles the Pallas kernels. libtpu 0.0.50 needs jax 0.11.2: newer jax refuses it.
if version("libtpu") == "0.0.50":
    packages += ["jax==0.11.2", "jaxlib==0.11.2"]
elif version("jax") is None:
    packages += ["jax"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)

# Use the package straight from the clone (no restart needed).
sys.path.insert(0, "w2v-bert-tpu/src")

for pkg in ["torch", "torch_tpu", "libtpu", "jax", "jaxlib", "transformers"]:
    print(f"{pkg:13} {version(pkg)}")

## 2. Load the model

`JAX_PLATFORMS=cpu` must be set before anything imports JAX: JAX only compiles the kernels,
and TorchTPU owns the TPU. Only one process can use the TPU, so close other notebooks that use it.

In [ ]:
import os

os.environ["JAX_PLATFORMS"] = "cpu"

import time

import torch

import w2vbert_tpu

# Downloads the weights, converts to BF16, moves to the TPU and sets up torch.compile.
model = w2vbert_tpu.load("facebook/w2v-bert-2.0")

## 3. Download the example audio

In [ ]:
from huggingface_hub import hf_hub_download
from IPython.display import Audio, display

files = [
    hf_hub_download("Yehor/w2v-bert-uk-v3", f"example_{i}.wav", repo_type="space")
    for i in range(1, 7)
]
clips = [w2vbert_tpu.load_audio(f) for f in files]  # (waveform, 16000)

for path, (audio, sr) in zip(files, clips):
    print(f"{os.path.basename(path)}: {audio.numel() / sr:.2f} s")
display(Audio(clips[0][0].numpy(), rate=clips[0][1]))

## 4. Run each clip

Every new input shape is compiled once (8–30 s). Padding each clip to a fixed size
(`pad_to_bucket`) lets all six clips share one compiled shape: the first call is slow,
the rest are fast.

In [ ]:
def embed(waveforms, sr):
    inputs = w2vbert_tpu.extract_features(waveforms, sampling_rate=sr)
    batch = w2vbert_tpu.pad_to_bucket(inputs["input_features"], inputs["attention_mask"])
    with torch.inference_mode():
        out = model(
            input_features=batch.input_features.to("tpu", torch.bfloat16),
            attention_mask=batch.attention_mask.to("tpu"),
        ).last_hidden_state
    torch.accelerator.synchronize()
    return w2vbert_tpu.unpad_output(out, batch.original_batch, batch.original_frames)


for path, (audio, sr) in zip(files, clips):
    t0 = time.perf_counter()
    hidden = embed(audio, sr)
    ms = (time.perf_counter() - t0) * 1000
    print(f"{os.path.basename(path)}: output {tuple(hidden.shape)} in {ms:,.1f} ms")

## 5. Run all clips as one batch

Clips of different lengths go in one batch; the attention mask keeps padding out of the result.
The first call compiles the batch-6 shape, the second shows the real speed.

In [ ]:
waveforms = [audio for audio, _ in clips]
for run in ("first (compiles)", "second"):
    t0 = time.perf_counter()
    hidden = embed(waveforms, 16000)
    print(f"{run}: output {tuple(hidden.shape)} in {(time.perf_counter() - t0) * 1000:,.1f} ms")

# Frames past each clip's real length are padding. Count the real frames per clip:
inputs = w2vbert_tpu.extract_features(waveforms, sampling_rate=16000)
print("real frames per clip:", inputs["attention_mask"].sum(dim=1).tolist())

## 6. Use the output

For example, a single vector per clip (the mean over real frames), and how similar the clips
are to each other.

In [ ]:
mask = inputs["attention_mask"].to(hidden.device, hidden.dtype).unsqueeze(-1)
vectors = (hidden * mask).sum(dim=1) / mask.sum(dim=1)  # [6, 1024]
vectors = torch.nn.functional.normalize(vectors.float().cpu(), dim=-1)
similarity = vectors @ vectors.T
print(similarity.numpy().round(3))